# Nomad access
Nomad has a REST HTTP API. You can use http requests to access all data.  
Nomad stores data in `.json` the units are stored in what is called [nomad metainfo](https://nomad-lab.eu/prod/v1/staging/gui/analyze/metainfo).  

Acessing nomad works with your account, if you dont have one create one here: https://nomad-lab.eu/prod/v1/staging/gui/about/information  

For this tutorial we will work with the central nomad, but keep in mind that the sol-ai nomad works the same. Just change the API link below.

## Authenticating
First we need to get an API token. Please use your nomad emali address from your account.  This token is used to access non public nomad entries. Since in this tutorial all entries are public we dont need it and keep the token empty. By providing your email you will get a token which you can use to access non public entries.

In [ ]:
import requests
import getpass

url = "https://nomad-lab.eu/prod/v1/api/v1"
email = ""
# url = "https://www.sol-ai.de.de/nomad-oasis/api/v1"

def get_token(url, name=None):
    user = name if name is not None else input("Username")
    print("Passwort: \n")
    password = getpass.getpass()
    # Get a token from the api, login
    response = requests.get(
        f'{url}/auth/token', params=dict(username=user, password=password))   
    return response.json()['access_token']

# get al entries related to this batch id
token = get_token(url,email) if email else ""
token

## Querying data
Now we can query specifc data. For example if we want to query data from the [Perovskite DB](https://nomad-lab.eu/prod/v1/staging/gui/search/solarcells) we can do the following. We use the token even so it is not needed, so you can use similar queries for non public data.  
This returns a list of 100 entries as python dictionaries. At the key `archive` it contains the entry point as in the gui: https://nomad-lab.eu/prod/v1/staging/gui/search/solarcells/entry/id/t3X--o-QsCZ7EzUsA3gnvTOmQJTX/data

In [ ]:
query = {
    'required': {
        'metadata': '*',
        'results': '*',
        'data': '*',
    },
    'owner': 'visible',
    'query': {'entry_type': "PerovskiteSolarCell"},
    'pagination': {
        'page_size': 10
    }
}
response = requests.post(f'{url}/entries/archive/query',
                         headers={'Authorization': f'Bearer {token}'}, json=query)
data = response.json()["data"]

The keys in the data section correspond to the sections in nomad: https://nomad-lab.eu/prod/v1/staging/gui/search/solarcells/entry/id/t3X--o-QsCZ7EzUsA3gnvTOmQJTX/data/data

In [ ]:
data[0]["archive"]["data"].keys()

Since the json contains only the values you can find the units in the metainfo, e.g. for voc it is in `volt` https://nomad-lab.eu/prod/v1/staging/gui/analyze/metainfo/perovskite_solar_cell_database/section_definitions@perovskite_solar_cell_database.schema_sections.jv.JV/forward_scan_Voc. There are also programmatic ways to access the units but since all archives of the same type are of the same unit we skip this here.

## Paging
As you see our query as a `page_size` of 100. Page sizes are limited by 10000. This deterimines how many entries are queried in one go. If you want to load the next `n` entries you can do the follwoing, the response contain next to the data a pagination object.

In [ ]:
print(response.json().keys())
pagination = response.json()["pagination"]
pagination

This can be used to get the next page by updating the query.

In [ ]:
query["pagination"]["page_after_value"] = pagination["next_page_after_value"]
response_next = requests.post(f'{url}/entries/query',
                         headers={'Authorization': f'Bearer {token}'}, json=query)
data_next = response_next.json()["data"]

## Accesing data
With this you can now write loops to iterate through data and plot it.

In [ ]:
query = {
    'required': {
        'metadata': '*',
        'results': '*',
        'data': '*',
    },
    'owner': 'visible',
    'query': {'entry_type': "PerovskiteSolarCell"},
    'pagination': {
        'page_size': 10
    }
}

efficiencies = []
vocs = []
i = 0
while len(efficiencies) < 30:
    print(f"Page {i}")
    response = requests.post(f'{url}/entries/archive/query',
                         headers={'Authorization': f'Bearer {token}'}, json=query)
    response_json = response.json()

    for data in response_json['data']:
        try:
            solar_cell = data["archive"]["results"]["properties"]["optoelectronic"]["solar_cell"]
        except:
            continue
        if "efficiency" in solar_cell and "open_circuit_voltage" in solar_cell:
            efficiencies.append(solar_cell["efficiency"])
            vocs.append(solar_cell["open_circuit_voltage"])

    next_value = response_json['pagination'].get('next_page_after_value')
    if not next_value:
        break
    query['pagination']['page_after_value'] = next_value
    i+=1

import matplotlib.pyplot as plt
plt.figure()
plt.plot(vocs, efficiencies, "x")
plt.xlabel("voc [V]")
plt.ylabel("eff [%]")
plt.show()

## Comment on nomads results, data and metadata sections
The archives right now are pretty big, because we include the `metadata`, `results` and `data` section. Since we only use the part of the `results` section which we need we can reduce the data by quite a lot. When we do not need the `data` section we can switch the end point to `/entries/query` instead of `/entries/archive/query` which is much faster (see next cell).  
The `results` section is provided by nomad and gives a definite strucutre for all entries in nomad. The `data` section is highly individual and can change from one entry to another.  Nomad then provides functionality to map data from the `data` section to the `results` section, which they call `normalization`. The `metadata`section contains general data on each entry, like `upload_id`, `entry_id` and `authorship`. You can find more information on using the API here: https://nomad-lab.eu/prod/v1/staging/docs/howto/programmatic/api.html

This even scales up to all entries, now 10000 entries per page til nothing left. We also load exactly those quantities which are needed by not loading all the data but only the part we are interested in.
This speeds up the process even more.

In [ ]:
# only results section
query = {
    'required': {
        "include":[
            'results.properties.optoelectronic.solar_cell.efficiency',
            'results.properties.optoelectronic.solar_cell.open_circuit_voltage'
        ]
    },
    'owner': 'visible',
    'query': {'entry_type': "PerovskiteSolarCell"},
    'pagination': {
        'page_size': 10000
    }
}

efficiencies = []
vocs = []
i = 0
while True:
    print(f"Page {i}")
    response = requests.post(f'{url}/entries/query',
                         headers={'Authorization': f'Bearer {token}'}, json=query)
    response_json = response.json()

    for data in response_json['data']:
        try:
            solar_cell = data["results"]["properties"]["optoelectronic"]["solar_cell"]
        except:
            continue
        if "efficiency" in solar_cell and "open_circuit_voltage" in solar_cell:
            efficiencies.append(solar_cell["efficiency"])
            vocs.append(solar_cell["open_circuit_voltage"])
        
    next_value = response_json['pagination'].get('next_page_after_value')
    if not next_value:
        break
    query['pagination']['page_after_value'] = next_value
    i+=1

import matplotlib.pyplot as plt
plt.figure()
plt.plot(vocs, efficiencies, "x")
plt.xlabel("voc [V]")
plt.ylabel("eff [%]")
plt.show()